In [ ]:
# Enlaces:
"""
1. Sala Especializada en Protección al Consumidor (SPC)
https://www.datosabiertos.gob.pe/dataset/expedientes-presentados-sala-de-protecci%C3%B3n-al-consumidor-indecopi-2018

https://www.datosabiertos.gob.pe/dataset/expedientes-presentados-sala-de-protecci%C3%B3n-al-consumidor-indecopi-2019

https://www.datosabiertos.gob.pe/dataset/expedientes-presentados-sala-de-protecci%C3%B3n-al-consumidor-indecopi-2020

https://www.datosabiertos.gob.pe/dataset/expedientes-presentados-sala-de-protecci%C3%B3n-al-consumidor-indecopi-2021

https://www.datosabiertos.gob.pe/dataset/indecopi-spc-expedientes-presentados


2. Comisiones de Protección al Consumidor (CC1, CC2, CC3)

https://www.datosabiertos.gob.pe/dataset/indecopi-cc1-expedientes-presentados

https://www.datosabiertos.gob.pe/dataset/indecopi-cc2-expedientes-presentados

https://www.datosabiertos.gob.pe/dataset/indecopi-cc3-expedientes-presentados


3. Órgano Resolutivo de Procedimientos Sumarísimos (OPS1, OPS2, OPS3)

https://www.datosabiertos.gob.pe/dataset/indecopi-ops1-expedientes-presentados

https://www.datosabiertos.gob.pe/dataset/indecopi-ops2-expedientes-presentados

https://www.datosabiertos.gob.pe/dataset/indecopi-ops3-expedientes-presentados

"""

# Importación de librerías

In [2]:
import pandas as pd
import glob
import os
import random
import numpy as np

# Carga de los datasets

In [3]:
# 1. CONFIGURACIÓN DE RUTAS
ruta_archivos = './Expedientes Presentados' 
todos_los_archivos = [f for f in glob.glob(os.path.join(ruta_archivos, "*.xls*")) 
                      if not os.path.basename(f).startswith("~$")]
data_list = []

In [4]:
print(f"--- Fase 1: Integración de {len(todos_los_archivos)} fuentes ---")

for archivo in todos_los_archivos:
    try:
        df = pd.read_excel(archivo)
        df.columns = [str(c).upper() for c in df.columns]
        
        temp_df = pd.DataFrame()

        # --- MAPEO DE COLUMNAS ORIGINALES ---
        f_col = [c for c in df.columns if any(x in c for x in ['FECHA', 'FEC_PRE', 'FECHA DE PRESENTACION', 'FECHA PRESENTACION', 'FECHA_PRESENTACION'])]
        if f_col: temp_df['fecha_raw'] = df[f_col[0]]

        t_col = [c for c in df.columns if any(x in c for x in ['TIPO_EXPEDIENTE', 'TIPO_PROCEDIMIENTO', 'TIPO DE EXP', 'VC_TIPO_EXPEDIENTE', 'TIPO EXPEDIENTE','TIPO DE EXP.'])]
        if t_col: temp_df['tipo_expediente'] = df[t_col[0]]

        m_col = [c for c in df.columns if any(x in c for x in ['MATERIA', 'SUB_SECTOR', 'SUBSECTOR', 'RUBRO', 'MATERIAS', 'SUB SECTOR', 'VC_RUBRO'])]
        if m_col: temp_df['materia'] = df[m_col[0]]

        d_col = [c for c in df.columns if any(x in c for x in ['DENUNCIADO', 'AUTORIDAD', 'DENUNCIADOS','DENUNCIADO(S)', 'ADMINISTRADOS'])]
        if d_col: temp_df['denunciado'] = df[d_col[0]]

        if not temp_df.empty:
            data_list.append(temp_df)
            print(f"Agregado: {os.path.basename(archivo)} ({len(temp_df)} filas)")
            
    except Exception as e:
        print(f"Error en {archivo}: {e}")

# Consolidación de base real
df_consolidado = pd.concat(data_list, ignore_index=True)

--- Fase 1: Integración de 18 fuentes ---
Agregado: Expedientes Presentados 2018.xlsx (3400 filas)
Agregado: Expedientes Presentados 2019.xlsx (3125 filas)
Agregado: Expedientes Presentados 2020.xlsx (2026 filas)
Agregado: Expedientes Presentados 2021.xlsx (2750 filas)
Agregado: INDECOPI_CC1_ExpedientesPresentados_2015_0.xlsx (2306 filas)
Agregado: INDECOPI_CC1_ExpedientesPresentados_2016_0.xlsx (2328 filas)
Agregado: INDECOPI_CC2_ExpedientesPresentados_2015.xlsx (2051 filas)
Agregado: INDECOPI_CC2_ExpedientesPresentados_2016.xlsx (2274 filas)
Agregado: INDECOPI_CC3_ExpedientesPresentados_2015.xlsx (115 filas)
Agregado: INDECOPI_CC3_ExpedientesPresentados_2016.xlsx (159 filas)
Agregado: INDECOPI_OPS1_ExpedientesPresentados_2015.xlsx (2363 filas)
Agregado: INDECOPI_OPS1_ExpedientesPresentados_2016.xlsx (2779 filas)
Agregado: INDECOPI_OPS2_ExpedientesPresentados_2015.xlsx (2317 filas)
Agregado: INDECOPI_OPS2_ExpedientesPresentados_2016.xlsx (2383 filas)
Agregado: INDECOPI_OPS3_Expediente

In [5]:
print(f"\n--- Fase 2: Data Augmentation (Asegurando >1M registros) ---")
current_len = len(df_consolidado)
factor_necesario = (1200000 // current_len) + 1
df_masivo = pd.concat([df_consolidado] * factor_necesario, ignore_index=True)
df_masivo = df_masivo.sample(frac=1).reset_index(drop=True)

print(f"Registros base: {current_len} | Factor: x{factor_necesario}")
print(f"Volumen alcanzado: {len(df_masivo)} registros")


--- Fase 2: Data Augmentation (Asegurando >1M registros) ---
Registros base: 43432 | Factor: x28
Volumen alcanzado: 1216096 registros


In [6]:
print(f"\n--- Fase 3: Enriquecimiento Técnico e Inyección de Anomalías ---")

departamentos_peru = [
    'LIMA', 'AREQUIPA', 'CUSCO', 'PIURA', 'CALLAO', 'JUNIN', 'LA LIBERTAD', 
    'LAMBAYEQUE', 'PUNO', 'ANCASH', 'ICA', 'LORETO', 'CAJAMARCA'
]
canales = ['WEB', 'APP_MOVIL', 'TELEFONICO', 'PRESENCIAL']
df_masivo['id_reclamo'] = range(1, len(df_masivo) + 1)
df_masivo['canal'] = [random.choice(canales) for _ in range(len(df_masivo))]

# GENERACIÓN SINTÉTICA DE REGIÓN
df_masivo['region'] = [random.choice(departamentos_peru) for _ in range(len(df_masivo))]

# Horas aleatorias HH:MM:SS
df_masivo['hora'] = [f"{random.randint(0,23):02d}:{random.randint(0,59):02d}:{random.randint(0,59):02d}" for _ in range(len(df_masivo))]

# --- INYECCIÓN DE RED FLAGS ---
# 1. Red Flag de Tiempo: 10,000 registros en el mismo segundo exacto
print("Inyectando Red Flag: 10,000 registros en un solo segundo (10:15:30)...")
idx_t = df_masivo.sample(n=10000).index
df_masivo.loc[idx_t, 'hora'] = "10:15:30"
df_masivo.loc[idx_t, 'canal'] = "WEB"
df_masivo.loc[idx_t, 'region'] = "LIMA" # Simula ataque masivo desde la capital
df_masivo.loc[idx_t, 'materia'] = "RECLAMO_BOT_TIME_ATTACK"

# 2. Red Flag de Texto: 8,000 registros con materia idéntica (Spam)
print("Inyectando Red Flag: Spam masivo de texto repetitivo...")
idx_x = df_masivo.sample(n=8000).index
df_masivo.loc[idx_x, 'materia'] = "SPAM_ADVERTISING_NON_SOLICITED_CONTENT"

# Consolidar Timestamp (YYYY-MM-DD HH:MM:SS)
df_masivo['timestamp'] = df_masivo['fecha_raw'].astype(str).str[:10] + " " + df_masivo['hora']

# Selección de columnas finales
df_final = df_masivo[[
    'id_reclamo', 'timestamp', 'tipo_expediente', 
    'materia', 'denunciado', 'canal', 'region'
]]

# Guardar a CSV
df_final.to_csv('dataset_indecopi_raw_+1M.csv', index=False)

print(f"\n--- PROCESO COMPLETADO ---")
print(f"Archivo generado: dataset_indecopi_raw_1M.csv")
print(f"Total registros: {len(df_final)}")


--- Fase 3: Enriquecimiento Técnico e Inyección de Anomalías ---
Inyectando Red Flag: 10,000 registros en un solo segundo (10:15:30)...
Inyectando Red Flag: Spam masivo de texto repetitivo...

--- PROCESO COMPLETADO ---
Archivo generado: dataset_indecopi_raw_1M.csv
Total registros: 1216096


In [7]:
# 1. Obtener todas las materias únicas detectadas
materias_reales = df_consolidado['materia'].unique().tolist()

# 2. Obtener todos los tipos de expediente únicos
tipos_reales = df_consolidado['tipo_expediente'].unique().tolist()

# 3. Obtener las combinaciones que más se repiten (para priorizar el banco de texto)
top_combinaciones = df_consolidado.groupby(['tipo_expediente', 'materia']).size().reset_index(name='cantidad').sort_values(by='cantidad', ascending=False)

print("--- REPORTE PARA GENERACIÓN DE TEXTO ---")
print(f"Materias encontradas ({len(materias_reales)}):", materias_reales)
print(f"Tipos de expediente ({len(tipos_reales)}):", tipos_reales)
print("\nTop 20 combinaciones existentes:")
print(top_combinaciones.head(20))

--- REPORTE PARA GENERACIÓN DE TEXTO ---
Materias encontradas (1025): ['SEGURO DE VIDA', 'CUENTA DE AHORROS', 'CREDITO DE CONSUMO', 'VENTA DE OTROS PRODUCTOS', 'TRANSPORTE DE PASAJEROS', 'PRESTAMO PERSONAL', 'SERVICIO DE ALIMENTACION Y BEBIDA', nan, 'CONSTRUCCION Y VENTA DE INMUEBLES', 'SERVICIOS DE ENSEÑANZA', 'VENTA DE VEHICULOS AUTOMOTORES Y MOTOCICLETAS', 'SEGURO DE SALUD', 'VENTA DE CALZADO Y ARTICULOS DE CUERO', 'CREDITO HIPOTECARIO', 'ELABORACION DE PRODUCTOS ALIMENTICIOS', 'SERVICIO DE HOSPEDAJE', 'OTROS SERVICIOS', 'SEGURO PATRIMONIAL', 'VENTA DE PRENDAS DE VESTIR', 'INTERVENCIONES QUIRURGICAS Y OTRAS /INTERVENCIONES QUIRURGICAS Y OTRAS', 'INTERVENCIONES QUIRURGICAS Y OTRAS /INTERVENCIONES QUIRURGICAS Y OTRAS /INTERVENCIONES QUIRURGICAS Y OTRAS', 'TARJETA DE CREDITO', 'TARJETA DE CREDITO /SEGUROS GENERALES', 'CUENTA DE AHORROS /SEGURO DE VIDA', 'REFINANCIAMIENTO', 'ACTIVIDADES DE AGENCIAS DE VIAJES', 'VENTA DE MEDICAMENTOS', 'ELABORACION DE BEBIDAS', 'OTROS SERVICIOS PARA EMPR

In [9]:
import pandas as pd
import os

# 1. Cargar el mapeo desde el CSV generado previamente
ruta_mapeo = 'mapeo_ia_materias.csv'

if os.path.exists(ruta_mapeo):
    print(f"--- Cargando mapeo semántico desde {ruta_mapeo} ---")
    df_mapeo_csv = pd.read_csv(ruta_mapeo)
    
    # Convertimos el DataFrame a un diccionario para un mapeo ultra rápido
    mapeo_dict = dict(zip(df_mapeo_csv['materia'], df_mapeo_csv['dominio']))
    
    # 2. Aplicar el mapeo al DataFrame masivo de forma instantánea
    print("Sincronizando dominios en el dataset masivo...")
    df_masivo['dominio'] = df_masivo['materia'].map(mapeo_dict).fillna('OTROS')
    
    # Verificación rápida
    conteo_dominios = df_masivo['dominio'].value_counts()
    print("\nResumen de dominios asignados:")
    print(conteo_dominios)

else:
    print(f"ERROR: No se encontró el archivo {ruta_mapeo}. ")
    print("Asegúrate de haber guardado el progreso de la clasificación anterior.")

--- Cargando mapeo semántico desde mapeo_ia_materias.csv ---
Sincronizando dominios en el dataset masivo...

Resumen de dominios asignados:
dominio
FINANZAS                             427122
OTROS                                203229
TRANSPORTES                          174567
RETAIL                               133301
INMOBILIARIO                         114366
EDUCACION                             65921
SALUD                                 33452
TELECOM                               32744
EDUCACIÓN                             17270
ADMINISTRATIVO                         8524
RETRO                                  2150
RETAIN                                 2096
SERVICIOS PROFESIONALES -> RETAIL       525
RETAIMOBILARIO                          415
CONSUMO                                 165
RETREAT                                 111
RETAIMOBILIARIO                          84
RETAIMOS                                 27
RETAIM                                   27
Name: count, dty

In [10]:
import requests
import pandas as pd
import json

In [ ]:
def pedir_texto_ia_detallado(dominio, materia, tipo):
    url = "http://localhost:11434/api/generate"
    
    # Prompt ultra-específico con triple contexto
    prompt = f"""
    Actúa como un ciudadano peruano molesto en INDECOPI.
    Escribe el cuerpo de un reclamo de 20 a 50 palabras.
    
    CONTEXTO:
    - Sector General: {dominio}
    - Materia Específica: {materia}
    - Tipo de Expediente: {tipo}
    
    REQUISITOS:
    - Usa lenguaje natural y directo.
    - Incluye jergas peruanas (ej. 'peloteo', 'estafa', 'pésimo', 'me pasean', 'una falta de respeto' , 'una decepción' , 'una traición' , 'una manipulación' , 'una desilusión' u otras jergas peruanas).
    - NO uses saludos ("Hola", "Estimados"), ni despedidas. Solo el problema.
    - Algunos reclamos hazlos parecidos hecho por otros usuarios para simular casos reales.
    - Algunos reclamos hazlos con lenguaje más formal pero igual de molestos, para simular variedad de perfiles.
    - Algunos recpalos hazlos con lenguaje más agresivo y otros con lenguaje más sutil pero igual de molestos, para simular variedad de perfiles.
    - Algunos reclamos hazlos parecer hechos por usuarios de provincias y otros por usuarios de Lima, para simular variedad regional.
    - Algunos reclamos hazlos con errores ortográficos y otros sin errores, para simular variedad de perfiles.
    - Algunos reclamos hazlos parecer hechos por IA para simular casos de reclamos generados por bots.
    """
    
    payload = {
        "model": "llama3",
        "prompt": prompt,
        "stream": False
    }
    
    try:
        response = requests.post(url, json=payload, timeout=20)
        return response.json()['response'].strip().replace('"', '')
    except:
        return f"Problema con {materia} en este {tipo}. Es una estafa y puro peloteo."

# 1. Obtener combinaciones únicas que existen en tu data masiva
print("Identificando combinaciones únicas (Materia x Tipo)...")
combinaciones_reales = df_masivo[['materia', 'tipo_expediente', 'dominio']].drop_duplicates()
banco_ia_triple = {}

print(f"--- Generando banco para {len(combinaciones_reales)} combinaciones ---")

# 2. Generar el banco (Esto toma tiempo, pero es oro puro para tu nota)
for i, row in combinaciones_reales.iterrows():
    key = (row['materia'], row['tipo_expediente'])
    # Generamos 3 variaciones por cada combinación específica
    banco_ia_triple[key] = [
        pedir_texto_ia_detallado(row['dominio'], row['materia'], row['tipo_expediente']) 
        for _ in range(3)
    ]
    if i % 10 == 0:
        print(f"Progreso: {i}/{len(combinaciones_reales)} combinaciones procesadas...")

Identificando combinaciones únicas (Materia x Tipo)...
--- Generando banco para 1573 combinaciones ---
Progreso: 0/1573 combinaciones procesadas...
Progreso: 10/1573 combinaciones procesadas...
Progreso: 20/1573 combinaciones procesadas...
Progreso: 30/1573 combinaciones procesadas...
Progreso: 60/1573 combinaciones procesadas...
Progreso: 70/1573 combinaciones procesadas...
Progreso: 110/1573 combinaciones procesadas...
Progreso: 150/1573 combinaciones procesadas...
Progreso: 160/1573 combinaciones procesadas...
Progreso: 210/1573 combinaciones procesadas...
Progreso: 220/1573 combinaciones procesadas...
Progreso: 300/1573 combinaciones procesadas...
Progreso: 330/1573 combinaciones procesadas...
Progreso: 460/1573 combinaciones procesadas...
Progreso: 530/1573 combinaciones procesadas...
Progreso: 570/1573 combinaciones procesadas...
Progreso: 740/1573 combinaciones procesadas...
Progreso: 770/1573 combinaciones procesadas...
Progreso: 860/1573 combinaciones procesadas...
Progreso: 9

In [12]:
import pandas as pd

# 1. Transformar el diccionario en una lista de registros planos
# banco_ia_triple tiene la estructura: {(materia, tipo): [texto1, texto2, texto3]}
datos_para_csv = []

print("Estructurando datos para el guardado...")

for (materia, tipo), textos in banco_ia_triple.items():
    # Buscamos el dominio correspondiente en tu mapeo previo
    # (Si mapeo_ia no está en memoria, puedes usar "GENERAL" como fallback)
    dominio_actual = mapeo_dict.get(materia, "OTROS")
    
    for i, txt in enumerate(textos):
        datos_para_csv.append({
            'dominio': dominio_actual,
            'materia': materia,
            'tipo_expediente': tipo,
            'variacion_id': i + 1,
            'texto_generado': str(txt) # Aseguramos que sea string para evitar errores previos
        })

# 2. Crear el DataFrame
df_banco_semillas = pd.DataFrame(datos_para_csv)

# 3. Guardar en CSV con codificación UTF-8 para jergas peruanas (tildes y ñ)
nombre_archivo = 'banco_semillas_ia_1573.csv'
df_banco_semillas.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')

print(f"--- ¡ÉXITO! ---")
print(f"Archivo guardado como: {nombre_archivo}")
print(f"Total de textos únicos respaldados: {len(df_banco_semillas)}")

Estructurando datos para el guardado...
--- ¡ÉXITO! ---
Archivo guardado como: banco_semillas_ia_1573.csv
Total de textos únicos respaldados: 4719


In [13]:
import random
import numpy as np

# 1. Función de asignación con corrección de tipos (evita el TypeError previo)
def asignar_texto_seguro(row):
    key = (row['materia'], row['tipo_expediente'])
    # Obtenemos las variantes generadas por la IA
    opciones = banco_ia_triple.get(key, ["Reclamo general por deficiencia en el servicio."])
    
    # Aseguramos que sea string y elegimos una variante al azar
    texto = str(random.choice(opciones))
    
    # Mutación ligera para que cada registro sea un objeto único en el Heap de Go
    if random.random() < 0.4:
        agregados = ["¡Es el colmo!", "Exijo solución.", "Ya basta de peloteo.", "Pésimo servicio."]
        texto = f"{texto} {random.choice(agregados)} [REF-{random.randint(100, 999)}]"
    
    return texto

# 2. Aplicar la asignación masiva
print("Sincronizando 1.2 millones de textos descriptivos...")
df_masivo['texto'] = df_masivo.apply(asignar_texto_seguro, axis=1)

# 3. Inyección de Red Flags (Anomalías para el detector en Go)
print("Inyectando Red Flags en la columna de texto...")

# A. Anomalía de Bot Attack (Mismo texto exacto en ráfaga)
# Usamos los mismos índices idx_t generados en tu fase de consolidación previa
idx_t = df_masivo[df_masivo['materia'] == "RECLAMO_BOT_TIME_ATTACK"].index
texto_bot = "SISTEMA_DETECTADO_ERROR_RECLAMO_MASIVO_AUTOMATIZADO_SEQUENCE_X99"
df_masivo.loc[idx_t, 'texto'] = texto_bot

# B. Anomalía de Spam / Contenido Malicioso
idx_x = df_masivo[df_masivo['materia'] == "SPAM_ADVERTISING_NON_SOLICITED_CONTENT"].index
texto_spam = "GANASTE UN PREMIO CLICK AQUI PARA RECLAMAR BITCOIN GRATIS ESTAFA TOTAL"
df_masivo.loc[idx_x, 'texto'] = texto_spam

# 4. Guardar Dataset Final Blindado
columnas_finales = [
    'id_reclamo', 'timestamp', 'tipo_expediente', 
    'materia', 'texto', 'denunciado', 'canal', 'region'
]

df_final = df_masivo[columnas_finales]
df_final.to_csv('dataset_indecopi_final_IA_con_redflags.csv', index=False, encoding='utf-8-sig')

print(f"--- FASE COMPLETADA ---")
print(f"Dataset generado: 'dataset_indecopi_final_IA_con_redflags.csv'")
print(f"Total registros: {len(df_final)}")

Sincronizando 1.2 millones de textos descriptivos...
Inyectando Red Flags en la columna de texto...
--- FASE COMPLETADA ---
Dataset generado: 'dataset_indecopi_final_IA_con_redflags.csv'
Total registros: 1216096
